# Lease Address Identification — LangGraph + OpenAI

End-to-end notebook:
- Dynamically locate the `dataset` folder.
- Find the input Excel workbook.
- Find lease PDFs recursively.
- **Upload every PDF through the OpenAI Files API (`/v1/files`) and capture each `file_id`.**
- **Map every Excel tenant `Name` to its PDF and `file_id`.**
- Match each tenant to a PDF by filename.
- Extract structured **`Name`, `Address` and `Confidence_Score`** for each Excel record.
- Validate the extracted name.
- **Compare the confidence score with a configurable threshold.**
- Write the address back to Excel and **colour each row**:
  - 🟩 **Green** – address found and `Confidence_Score >= CONFIDENCE_THRESHOLD`
  - 🟥 **Red** – confidence below the threshold, address not found, name mismatch, or no PDF / upload / extraction error
- Save `mock_lease_data_completed.xlsx`.

Recommended PDF names:
`lease_001_Emily_Carter.pdf`, `lease_002_Michael_Brown.pdf`, etc.

All tunable settings (threshold, model, which rows to process, etc.) live in **Cell 3**.

In [2]:
# Cell 1 — Install dependencies
# Run this cell once if packages are not installed.

%pip install -U openai langgraph pandas openpyxl python-dotenv


Note: you may need to restart the kernel to use updated packages.


d:\AddressIdentification\.venv\Scripts\python.exe: No module named pip


In [3]:
# Cell 2 — Imports

import os
import json
import re
from pathlib import Path
from typing import TypedDict, Optional, Any

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langgraph.graph import StateGraph, END
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

In [4]:
# Cell 3 — Configuration

DATASET_FOLDER_NAME = "dataset"
OUTPUT_EXCEL_NAME = "mock_lease_data_completed.xlsx"

# Change this if your environment uses another available model.
OPENAI_MODEL = "gpt-5.6-luna"

# ---- NEW: confidence / colouring ------------------------------------------
# Minimum LLM confidence (0.0 - 1.0) for a row to be GREEN.
CONFIDENCE_THRESHOLD = 0.80

GREEN_HEX = "C6EFCE"   # address found, confidence >= threshold
RED_HEX = "FFC7CE"     # low confidence / not found / error

# ---- NEW: behaviour switches ----------------------------------------------
# False -> process EVERY Excel record (matches the requirement).
# True  -> original behaviour: only rows whose Address is blank.
PROCESS_ONLY_MISSING_ADDRESSES = False

# If a row already has an Address, replace it with the LLM address on GREEN?
# False -> keep the existing address (LLM result is still shown in Extracted_Address).
OVERWRITE_EXISTING_ADDRESS = False

# Require the Name returned by the LLM to match the Excel Name (existing check).
ENFORCE_NAME_VALIDATION = True

# Delete the uploaded PDFs from OpenAI storage when the run ends
# (lease PDFs contain personal data).
DELETE_UPLOADED_FILES_AFTER_RUN = False

# ---- NEW: extra audit columns written to the output workbook ---------------
EXTRACTED_ADDRESS_COL = "Extracted_Address"
CONFIDENCE_COL = "Confidence_Score"
STATUS_COL = "Status"

print("Configuration loaded.")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

Configuration loaded.
Confidence threshold: 0.8


In [5]:
# Cell 4 — Initialize OpenAI client

def initialize_openai():
    load_dotenv()

    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise ValueError(
            "OPENAI_API_KEY was not found. "
            "Add OPENAI_API_KEY=... to your .env file or environment."
        )

    return OpenAI(api_key=api_key)


client = initialize_openai()
print("OpenAI client initialized successfully.")


OpenAI client initialized successfully.


In [6]:
# Cell 5 — Find dataset directory dynamically

def find_dataset_directory(
    start_dir: Optional[Path] = None,
    dataset_name: str = DATASET_FOLDER_NAME,
) -> Path:
    current = (start_dir or Path.cwd()).resolve()

    while True:
        candidate = current / dataset_name

        if candidate.is_dir():
            return candidate

        if current.parent == current:
            break

        current = current.parent

    raise FileNotFoundError(
        f"Could not find '{dataset_name}' starting from "
        f"{start_dir or Path.cwd()}"
    )


DATASET_DIR = find_dataset_directory()
print(f"Dataset directory: {DATASET_DIR}")


Dataset directory: D:\AddressIdentification\dataset


In [13]:
# Cell 6 — Find input Excel file

def find_excel_file(dataset_dir: Path) -> Path:
    files = [
        p for p in dataset_dir.glob("*.xlsx")
        if not p.name.endswith("_completed.xlsx")
    ]

    if not files:
        raise FileNotFoundError(
            f"No input .xlsx file found in {dataset_dir}"
        )

    # If multiple Excel files exist, keep the first one
    # and delete the remaining Excel files.
    if len(files) > 1:
        input_excel = files[1]
        print(input_excel)
        for excel_file in files[1:]:
            excel_file.unlink()
            print(f"Deleted extra Excel file: {excel_file}")

        return input_excel

    return files[0]


INPUT_EXCEL = find_excel_file(DATASET_DIR)
OUTPUT_EXCEL = DATASET_DIR / OUTPUT_EXCEL_NAME

print(f"Input Excel : {INPUT_EXCEL}")
print(f"Output Excel: {OUTPUT_EXCEL}")

Input Excel : D:\AddressIdentification\dataset\mock_lease_data.xlsx
Output Excel: D:\AddressIdentification\dataset\mock_lease_data_completed.xlsx


In [9]:
# Cell 7 — Find PDF files

def find_pdf_files(dataset_dir: Path) -> list[Path]:
    pdf_files = sorted(dataset_dir.rglob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found under {dataset_dir}"
        )

    return pdf_files


PDF_FILES = find_pdf_files(DATASET_DIR)

print(f"Found {len(PDF_FILES)} PDF files.")
for pdf in PDF_FILES:
    print(f" - {pdf}")


Found 7 PDF files.
 - D:\AddressIdentification\dataset\ML-2025-1001_Emily_Carter.pdf
 - D:\AddressIdentification\dataset\ML-2025-1002_Michael_Brown.pdf
 - D:\AddressIdentification\dataset\ML-2025-1003_Sarah_Wilson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1004_Daniel_Martinez.pdf
 - D:\AddressIdentification\dataset\ML-2025-1005_Olivia_Thompson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1006_James_Anderson.pdf
 - D:\AddressIdentification\dataset\ML-2025-1007_Sophia_Lewis.pdf


In [10]:
# Cell 8 — Read and validate Excel

def read_excel_file(excel_path: Path) -> pd.DataFrame:
    df = pd.read_excel(excel_path)

    required_columns = ["Name", "DOB", "Address"]
    missing = [c for c in required_columns if c not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required Excel columns: {missing}"
        )

    # An empty Address column is read as float64; make it text-safe so
    # strings can be written back without dtype errors.
    df["Address"] = df["Address"].astype("object")

    # Audit columns used by the confidence / colouring logic.
    for col in (EXTRACTED_ADDRESS_COL, CONFIDENCE_COL, STATUS_COL):
        if col not in df.columns:
            df[col] = pd.Series([None] * len(df), dtype="object")
        else:
            df[col] = df[col].astype("object")

    return df


df = read_excel_file(INPUT_EXCEL)

print(f"Total records: {len(df)}")
display(df.head())

Total records: 7


,Name,DOB,Address,Lease ID,Lease Start Date,Lease End Date,Monthly Rent,Property Type,Lease Status,Tenant Phone,Tenant Email,Extracted_Address,Confidence_Score,Status
0,Emily Carter,05/14/1992,"125 Oak Street, Dallas, TX 75201",ML-2025-1001,2025-01-01,2025-12-31,1850,Apartment,Active,555-200-4100,emily.carter@example.com,None,None,None
1,Michael Brown,11/23/1988,NaN,ML-2025-1002,2025-01-14,2026-01-13,2150,Townhouse,Active,555-201-4101,michael.brown@example.com,None,None,None
2,Sarah Wilson,07/09/1995,"318 Cedar Lane, Houston, TX 77002",ML-2025-1003,2025-01-27,2026-01-26,1725,Condominium,Active,555-202-4102,sarah.wilson@example.com,None,None,None
3,Daniel Martinez,02/18/1990,NaN,ML-2025-1004,2025-02-09,2026-02-08,2400,Single-Family Home,Active,555-203-4103,daniel.martinez@example.com,None,None,None
4,Olivia Thompson,09/27/1993,NaN,ML-2025-1005,2025-02-22,2026-02-21,1950,Apartment,Active,555-204-4104,olivia.thompson@example.com,None,None,None


In [11]:
# Cell 9 — Identify rows with missing addresses / rows to process

def get_missing_address_indexes(df: pd.DataFrame) -> list:
    mask = (
        df["Address"].isna()
        | df["Address"].fillna("").astype(str).str.strip().eq("")
    )
    return df.index[mask].tolist()


def get_records_to_process(df: pd.DataFrame) -> list:
    """All records by default; only blank-address rows if configured."""
    if PROCESS_ONLY_MISSING_ADDRESSES:
        return get_missing_address_indexes(df)
    return df.index.tolist()


missing_indexes = get_missing_address_indexes(df)

print(f"Missing-address records: {len(missing_indexes)}")
print(f"Indexes: {missing_indexes}")
print(f"Records that will be processed: {len(get_records_to_process(df))}")

Missing-address records: 3
Indexes: [1, 3, 4]
Records that will be processed: 7


In [12]:
# Cell 10 — Normalization helpers

def normalize_name(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def normalize_dob(value) -> str:
    if value is None or pd.isna(value):
        return ""

    try:
        dt = pd.to_datetime(value)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return re.sub(r"[^0-9]", "", str(value))


NOT_FOUND_TOKENS = {
    "", "none", "null", "nan", "n/a", "na",
    "not found", "not available", "unknown",
}


def is_address_missing(value) -> bool:
    """True for None / NaN / blank / 'not found'-style placeholder text."""
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower() in NOT_FOUND_TOKENS

In [13]:
# Cell 11 — Match tenant to PDF

def find_matching_pdf(
    tenant_name: str,
    tenant_dob,
    pdf_files: list[Path],
) -> Optional[Path]:
    name_key = normalize_name(tenant_name)
    dob_key = normalize_dob(tenant_dob)

    candidates = []

    for pdf in pdf_files:
        filename_key = normalize_name(pdf.stem)

        if name_key and name_key in filename_key:
            candidates.append(pdf)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1 and dob_key:
        dob_candidates = [
            p for p in candidates
            if dob_key in re.sub(r"[^0-9]", "", p.stem)
        ]

        if len(dob_candidates) == 1:
            return dob_candidates[0]

    if len(candidates) > 1:
        names = ", ".join(p.name for p in candidates)
        raise ValueError(
            f"Multiple PDFs matched '{tenant_name}': {names}"
        )

    return None


In [14]:
# Cell 12 — Upload PDF through /v1/files

def upload_pdf(client: OpenAI, pdf_path: Path) -> str:
    with pdf_path.open("rb") as file:
        uploaded = client.files.create(
            file=file,
            purpose="user_data",
        )

    print(f"Uploaded {pdf_path.name} -> {uploaded.id}")

    return uploaded.id


In [17]:
# Cell 13 — Upload all PDFs and map file_id -> tenant Name   (NEW)

def upload_all_pdfs(client: OpenAI, pdf_files: list[Path]) -> dict:
    """Upload every PDF once. Returns {str(pdf_path): file_id or None}."""
    uploaded = {}

    for pdf in pdf_files:
        try:
            uploaded[str(pdf)] = upload_pdf(client, pdf)
        except Exception as exc:
            print(f"Upload FAILED for {pdf.name}: {exc}")
            uploaded[str(pdf)] = None

    return uploaded


def build_file_map(
    df: pd.DataFrame,
    record_indexes: list,
    pdf_files: list[Path],
    uploaded: dict,
) -> dict:
    """
    Map each Excel record (row index) to its tenant Name, PDF and OpenAI file_id.
    Keyed by row index so two tenants with the same name cannot collide.
    """
    file_map = {}

    for index in record_indexes:
        row = df.loc[index]
        name = str(row["Name"]).strip()

        entry = {
            "name": name,
            "pdf_path": None,
            "file_id": None,
            "error": None,
        }

        try:
            pdf = find_matching_pdf(name, row["DOB"], pdf_files)

            if pdf is None:
                entry["error"] = "No PDF filename match"
            else:
                entry["pdf_path"] = str(pdf)
                entry["file_id"] = uploaded.get(str(pdf))

                if not entry["file_id"]:
                    entry["error"] = "PDF upload failed"

        except ValueError as exc:
            entry["error"] = str(exc)

        file_map[index] = entry

    return file_map


def delete_uploaded_files(client: OpenAI, uploaded: dict) -> None:
    for pdf_path, file_id in uploaded.items():
        if not file_id:
            continue
        try:
            client.files.delete(file_id)
            print(f"Deleted {file_id} ({Path(pdf_path).name})")
        except Exception as exc:
            print(f"Could not delete {file_id}: {exc}")

In [18]:
# Cell 14 — Structured output schema

LEASE_SCHEMA = {
    "type": "object",
    "properties": {
        "Name": {
            "type": "string",
            "description": "Tenant name as written in the lease."
        },
        "Address": {
            "type": ["string", "null"],
            "description": (
                "Complete tenant/property address from the lease, "
                "or null if no address is present."
            )
        },
        "Confidence_Score": {
            "type": "number",
            "description": (
                "Confidence between 0.0 and 1.0 that Name and Address are "
                "correct and belong to the expected tenant. "
                "Use 0.0 when no address is found."
            )
        }
    },
    "required": ["Name", "Address", "Confidence_Score"],
    "additionalProperties": False,
}

In [19]:
# Cell 15 — Extract Name + Address + Confidence_Score from PDF

def extract_name_and_address(
    client: OpenAI,
    file_id: str,
    expected_name: str,
    expected_dob,
) -> dict:

    prompt = f'''
Extract the tenant's Name, complete Address and a Confidence_Score from this lease PDF.

Expected tenant from Excel:
Name: {expected_name}
DOB: {expected_dob}

Rules:
- Extract information only from the PDF.
- Do not invent or infer an address that is not present.
- Return the complete tenant/property address.
- The returned name must correspond to the expected tenant.
- If multiple addresses occur, choose the tenant/property address.
- If no address can be found, return Address as null and Confidence_Score as 0.0.

Confidence_Score (float from 0.0 to 1.0):
- 0.90-1.00: address is explicitly labelled as the tenant/property address and the tenant name matches exactly.
- 0.70-0.89: address is clearly present but the label or tenant match is slightly ambiguous.
- 0.40-0.69: several candidate addresses, partial address, or unclear text.
- 0.00-0.39: guess, heavily incomplete, or not found.
'''

    response = client.responses.create(
        model=OPENAI_MODEL,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_file",
                        "file_id": file_id,
                    },
                    {
                        "type": "input_text",
                        "text": prompt,
                    },
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "lease_information",
                "schema": LEASE_SCHEMA,
                "strict": True,
            }
        },
    )

    if not response.output_text:
        raise ValueError("OpenAI returned an empty response.")

    data = json.loads(response.output_text)

    # Normalise: clamp the score to [0, 1] and turn "not found" text into None.
    try:
        score = float(data.get("Confidence_Score", 0.0))
    except (TypeError, ValueError):
        score = 0.0
    score = max(0.0, min(1.0, score))

    address = data.get("Address")
    address = None if is_address_missing(address) else str(address).strip()

    return {
        "Name": str(data.get("Name") or "").strip(),
        "Address": address,
        "Confidence_Score": score,
    }

In [20]:
# Cell 16 — Validate extracted tenant name

def validate_extracted_name(
    expected_name: str,
    extracted_name: str,
) -> bool:

    expected = normalize_name(expected_name)
    extracted = normalize_name(extracted_name)

    valid = expected == extracted

    print(
        f"Name validation: {'PASSED' if valid else 'FAILED'} | "
        f"Excel='{expected_name}' | PDF='{extracted_name}'"
    )

    return valid


In [21]:
# Cell 17 — LangGraph state

class LeaseState(TypedDict, total=False):
    dataframe: Any
    pdf_files: list

    # run-level (set once by prepare_files)
    record_indexes: list
    cursor: int
    uploaded_files: dict          # {pdf_path: file_id}
    file_map: dict                # {row_index: {name, pdf_path, file_id, error}}
    row_results: dict             # {row_index: "GREEN" | "RED"}

    # current record
    current_index: int
    tenant_name: str
    tenant_dob: str
    pdf_path: Optional[str]
    file_id: Optional[str]
    extracted_name: Optional[str]
    extracted_address: Optional[str]
    confidence_score: Optional[float]
    result: Optional[str]         # "GREEN" | "RED"
    status: Optional[str]         # human-readable reason written to Excel
    error: Optional[str]

    # counters
    processed_count: int
    green_count: int
    red_count: int

In [22]:
# Cell 18 — LangGraph node: upload all PDFs and build the file_id map   (NEW)

def node_prepare_files(state: LeaseState):
    df = state["dataframe"]
    record_indexes = get_records_to_process(df)

    uploaded = upload_all_pdfs(client, state["pdf_files"])
    file_map = build_file_map(df, record_indexes, state["pdf_files"], uploaded)

    mapping_table = pd.DataFrame(
        [
            {
                "Row": idx,
                "Name": e["name"],
                "PDF": Path(e["pdf_path"]).name if e["pdf_path"] else None,
                "file_id": e["file_id"],
                "Issue": e["error"],
            }
            for idx, e in file_map.items()
        ]
    )
    print("\nName -> PDF -> OpenAI file_id mapping:")
    print(mapping_table.to_string(index=False))
    print()

    return {
        "record_indexes": record_indexes,
        "cursor": 0,
        "uploaded_files": uploaded,
        "file_map": file_map,
        "row_results": {},
    }

In [23]:
# Cell 19 — LangGraph node: get next record

def node_get_next_record(state: LeaseState):
    # Cursor-based iteration: every record is visited exactly once, even if
    # it ends up RED (the old "first blank Address" lookup re-selected skipped
    # rows forever).
    cursor = state.get("cursor", 0)
    record_indexes = state["record_indexes"]

    if cursor >= len(record_indexes):
        return {"current_index": -1}

    index = record_indexes[cursor]
    row = state["dataframe"].loc[index]

    return {
        "current_index": index,
        "cursor": cursor + 1,
        "tenant_name": str(row["Name"]).strip(),
        "tenant_dob": str(row["DOB"]).strip(),
        # reset per-record values so nothing leaks from the previous row
        "pdf_path": None,
        "file_id": None,
        "extracted_name": None,
        "extracted_address": None,
        "confidence_score": None,
        "result": None,
        "status": None,
        "error": None,
    }

In [24]:
# Cell 20 — LangGraph node: look up the PDF / file_id for this record

def node_lookup_file(state: LeaseState):
    entry = state["file_map"].get(state["current_index"])

    if not entry or not entry.get("file_id"):
        reason = (entry or {}).get("error") or "No uploaded file for this record"
        return {
            "pdf_path": (entry or {}).get("pdf_path"),
            "error": f"{reason} ({state['tenant_name']})",
        }

    print(
        f"{state['tenant_name']}: "
        f"{Path(entry['pdf_path']).name} -> {entry['file_id']}"
    )

    return {
        "pdf_path": entry["pdf_path"],
        "file_id": entry["file_id"],
        "error": None,
    }

In [25]:
# Cell 21 — LangGraph node: extract data

def node_extract_data(state: LeaseState):
    if not state.get("file_id"):
        return {
            "error": state.get(
                "error",
                "File ID is missing",
            )
        }

    try:
        result = extract_name_and_address(
            client=client,
            file_id=state["file_id"],
            expected_name=state["tenant_name"],
            expected_dob=state["tenant_dob"],
        )

        print(json.dumps(result, indent=2))

        return {
            "extracted_name": result["Name"],
            "extracted_address": result["Address"],
            "confidence_score": result["Confidence_Score"],
            "error": None,
        }

    except Exception as exc:
        return {
            "error": f"Extraction failed: {exc}"
        }

In [26]:
# Cell 22 — LangGraph node: evaluate result (validation + confidence threshold)

def node_evaluate(state: LeaseState):
    if state.get("error"):
        return {"result": "RED", "status": f"RED - {state['error']}"}

    address = state.get("extracted_address")
    score = state.get("confidence_score") or 0.0

    # 1) Address could not be found -> RED
    if is_address_missing(address):
        return {"result": "RED", "status": "RED - Address not found"}

    # 2) Extracted name must belong to the Excel tenant (existing check)
    if ENFORCE_NAME_VALIDATION and not validate_extracted_name(
        state["tenant_name"],
        state.get("extracted_name"),
    ):
        return {
            "result": "RED",
            "status": f"RED - Name mismatch (PDF: {state.get('extracted_name')})",
        }

    # 3) Confidence below threshold -> RED
    if score < CONFIDENCE_THRESHOLD:
        return {
            "result": "RED",
            "status": (
                f"RED - Low confidence "
                f"({score:.2f} < {CONFIDENCE_THRESHOLD:.2f})"
            ),
        }

    # 4) Found and confident enough -> GREEN
    return {
        "result": "GREEN",
        "status": f"GREEN - Address found ({score:.2f} >= {CONFIDENCE_THRESHOLD:.2f})",
    }

In [27]:
# Cell 23 — LangGraph node: update Excel (DataFrame)

def node_update_excel(state: LeaseState):
    df = state["dataframe"]
    index = state["current_index"]
    result = state["result"]

    # Audit columns are always filled so RED rows can be reviewed by a human.
    df.at[index, STATUS_COL] = state["status"]
    df.at[index, CONFIDENCE_COL] = state.get("confidence_score")
    df.at[index, EXTRACTED_ADDRESS_COL] = state.get("extracted_address")

    # The real Address column is only filled for GREEN rows.
    if result == "GREEN" and (
        is_address_missing(df.at[index, "Address"])
        or OVERWRITE_EXISTING_ADDRESS
    ):
        df.at[index, "Address"] = state["extracted_address"]
        print(f"UPDATED: {state['tenant_name']} -> {state['extracted_address']}")
    else:
        print(f"{result}: {state['tenant_name']} | {state['status']}")

    row_results = dict(state.get("row_results", {}))
    row_results[index] = result

    return {
        "dataframe": df,
        "row_results": row_results,
        "processed_count": state.get("processed_count", 0) + 1,
        "green_count": state.get("green_count", 0) + (result == "GREEN"),
        "red_count": state.get("red_count", 0) + (result == "RED"),
    }

In [28]:
# Cell 24 — LangGraph routing

def route_after_update(state: LeaseState):
    if state.get("cursor", 0) < len(state["record_indexes"]):
        return "next_record"

    return "save"

In [29]:
# Cell 25 — LangGraph node: save Excel and colour rows

def apply_row_colors(excel_path: Path, df: pd.DataFrame, row_results: dict) -> None:
    fills = {
        "GREEN": PatternFill(start_color=GREEN_HEX, end_color=GREEN_HEX, fill_type="solid"),
        "RED": PatternFill(start_color=RED_HEX, end_color=RED_HEX, fill_type="solid"),
    }

    wb = load_workbook(excel_path)
    ws = wb.active

    for index, result in row_results.items():
        excel_row = df.index.get_loc(index) + 2   # +1 header, +1 one-based
        for col in range(1, ws.max_column + 1):
            ws.cell(row=excel_row, column=col).fill = fills[result]

    wb.save(excel_path)


def node_save_excel(state: LeaseState):
    df = state["dataframe"]

    df.to_excel(OUTPUT_EXCEL, index=False)
    apply_row_colors(OUTPUT_EXCEL, df, state.get("row_results", {}))

    print(f"Saved: {OUTPUT_EXCEL}")
    print(f"Processed: {state.get('processed_count', 0)}")
    print(f"Green    : {state.get('green_count', 0)}")
    print(f"Red      : {state.get('red_count', 0)}")

    if DELETE_UPLOADED_FILES_AFTER_RUN:
        delete_uploaded_files(client, state.get("uploaded_files", {}))

    return {}

In [30]:
# Cell 26 — Build LangGraph

def build_lease_graph():
    graph = StateGraph(LeaseState)

    graph.add_node("prepare_files", node_prepare_files)
    graph.add_node("get_next_record", node_get_next_record)
    graph.add_node("lookup_file", node_lookup_file)
    graph.add_node("extract_data", node_extract_data)
    graph.add_node("evaluate", node_evaluate)
    graph.add_node("update_excel", node_update_excel)
    graph.add_node("save_excel", node_save_excel)

    graph.set_entry_point("prepare_files")
    graph.add_edge("prepare_files", "get_next_record")

    graph.add_conditional_edges(
        "get_next_record",
        lambda state: (
            "finish"
            if state.get("current_index") == -1
            else "continue"
        ),
        {
            "continue": "lookup_file",
            "finish": "save_excel",
        },
    )

    graph.add_edge("lookup_file", "extract_data")
    graph.add_edge("extract_data", "evaluate")
    graph.add_edge("evaluate", "update_excel")

    graph.add_conditional_edges(
        "update_excel",
        route_after_update,
        {
            "next_record": "get_next_record",
            "save": "save_excel",
        },
    )

    graph.add_edge("save_excel", END)

    return graph.compile()

In [31]:
# Cell 27 — Run complete workflow

app = build_lease_graph()

initial_state: LeaseState = {
    "dataframe": df.copy(),
    "pdf_files": PDF_FILES,
    "processed_count": 0,
    "green_count": 0,
    "red_count": 0,
}

# LangGraph's default recursion_limit is 25 steps; each record uses 5 nodes,
# so scale the limit with the number of records.
recursion_limit = 20 + 5 * len(get_records_to_process(df))

result = app.invoke(
    initial_state,
    config={"recursion_limit": recursion_limit},
)

print("\n========================================")
print("PIPELINE COMPLETED")
print("========================================")
print(f"Output: {OUTPUT_EXCEL}")

Uploaded ML-2025-1001_Emily_Carter.pdf -> file-R8Px5EfXAvgZeNy97TVwxH
Uploaded ML-2025-1002_Michael_Brown.pdf -> file-WGd2GhRsTxZiKXwfrFuRm9
Uploaded ML-2025-1003_Sarah_Wilson.pdf -> file-AM7GRMtnDsBSJu3KtA9ghp
Uploaded ML-2025-1004_Daniel_Martinez.pdf -> file-NAwhNRVae2cocjGUdZSYGN
Uploaded ML-2025-1005_Olivia_Thompson.pdf -> file-RjRmpfBZySkmS57kGwuVHL
Uploaded ML-2025-1006_James_Anderson.pdf -> file-HXa78WaSJxKDs8g28fHXnT
Uploaded ML-2025-1007_Sophia_Lewis.pdf -> file-YRNEV5pVQCBBnZ561RdtZg

Name -> PDF -> OpenAI file_id mapping:
 Row            Name                              PDF                     file_id Issue
   0    Emily Carter    ML-2025-1001_Emily_Carter.pdf file-R8Px5EfXAvgZeNy97TVwxH  None
   1   Michael Brown   ML-2025-1002_Michael_Brown.pdf file-WGd2GhRsTxZiKXwfrFuRm9  None
   2    Sarah Wilson    ML-2025-1003_Sarah_Wilson.pdf file-AM7GRMtnDsBSJu3KtA9ghp  None
   3 Daniel Martinez ML-2025-1004_Daniel_Martinez.pdf file-NAwhNRVae2cocjGUdZSYGN  None
   4 Olivia Thompson 

In [32]:
# Cell 28 — Verify completed Excel (data + row colours)

completed_df = pd.read_excel(OUTPUT_EXCEL)

remaining_missing = (
    completed_df["Address"].isna()
    | completed_df["Address"].fillna("").astype(str).str.strip().eq("")
).sum()

print(f"Remaining missing addresses: {remaining_missing}")

# Read the fill colour of the first cell of every data row.
ws = load_workbook(OUTPUT_EXCEL).active
color_names = {GREEN_HEX: "GREEN", RED_HEX: "RED"}

row_colors = []
for r in range(2, ws.max_row + 1):
    rgb = (ws.cell(row=r, column=1).fill.start_color.rgb or "")[-6:]
    row_colors.append(color_names.get(rgb, "NONE"))

completed_df["Row_Color"] = row_colors

print(completed_df["Row_Color"].value_counts().to_string())

display(completed_df)

Remaining missing addresses: 0
Row_Color
GREEN    7


,Name,DOB,Address,Lease ID,Lease Start Date,Lease End Date,Monthly Rent,Property Type,Lease Status,Tenant Phone,Tenant Email,Extracted_Address,Confidence_Score,Status,Row_Color
0,Emily Carter,05/14/1992,"125 Oak Street, Dallas, TX 75201",ML-2025-1001,2025-01-01,2025-12-31,1850,Apartment,Active,555-200-4100,emily.carter@example.com,"125 Oak Street, Dallas, TX 75201",1,GREEN - Address found (1.00 >= 0.80),GREEN
1,Michael Brown,11/23/1988,"742 Pine Avenue, Austin, TX 78701",ML-2025-1002,2025-01-14,2026-01-13,2150,Townhouse,Active,555-201-4101,michael.brown@example.com,"742 Pine Avenue, Austin, TX 78701",1,GREEN - Address found (1.00 >= 0.80),GREEN
2,Sarah Wilson,07/09/1995,"318 Cedar Lane, Houston, TX 77002",ML-2025-1003,2025-01-27,2026-01-26,1725,Condominium,Active,555-202-4102,sarah.wilson@example.com,"318 Cedar Lane, Houston, TX 77002",1,GREEN - Address found (1.00 >= 0.80),GREEN
3,Daniel Martinez,02/18/1990,"904 Maple Drive, Phoenix, AZ 85004",ML-2025-1004,2025-02-09,2026-02-08,2400,Single-Family Home,Active,555-203-4103,daniel.martinez@example.com,"904 Maple Drive, Phoenix, AZ 85004",1,GREEN - Address found (1.00 >= 0.80),GREEN
4,Olivia Thompson,09/27/1993,"156 Willow Road, Denver, CO 80202",ML-2025-1005,2025-02-22,2026-02-21,1950,Apartment,Active,555-204-4104,olivia.thompson@example.com,"156 Willow Road, Denver, CO 80202",1,GREEN - Address found (1.00 >= 0.80),GREEN
5,James Anderson,12/05/1986,"481 Birch Court, Seattle, WA 98101",ML-2025-1006,2025-03-07,2026-03-06,2250,Condominium,Active,555-205-4105,james.anderson@example.com,"481 Birch Court, Seattle, WA 98101",1,GREEN - Address found (1.00 >= 0.80),GREEN
6,Sophia Lewis,03/21/1997,"229 Aspen Way, Portland, OR 97205",ML-2025-1007,2025-03-20,2026-03-19,2100,Townhouse,Active,555-206-4106,sophia.lewis@example.com,"229 Aspen Way, Portland, OR 97205",1,GREEN - Address found (1.00 >= 0.80),GREEN
